# scgraph 4.0: Global Intermodal Network & Acceptance Demonstration

This notebook demonstrates the global routing capabilities and Stage 4 acceptance validation of **scgraph 4.0**.

### Key Capabilities Demonstrated:
1. **Planetary-Scale Intermodal Graph (`world_highways_and_marnet`)**:
   - Over 572,000 nodes and 1.68 million directed arcs combining global highway systems, car ferries, port connectors, and maritime shipping lanes.
2. **Zero-RAM Metadata Inspection (`GeoGraph.get_scgraph_info`)**:
   - Instantaneous inspection of binary `.scgraph` headers, spatial bounds, coordinate precision (`int32` at 10⁻⁷ degrees), and source attribution.
3. **Stage 4 Acceptance Certification (`validate_network`)**:
   - Execution of declarative test cases across multiple continents and oceans with tight distance tolerances.
4. **Monomodal vs. Multimodal Routing**:
   - Comparison between road-only routing (`world_highways`) and intermodal routing (`world_highways_and_marnet`).
5. **Interactive Global Cartography (Folium)**:
   - High-resolution interactive world map displaying certified corridors spanning Europe, the Americas, Asia, the Atlantic, and the Pacific.

In [1]:
import os
import folium
from scgraph import GeoGraph
from scgraph.pipelines.steps.stage4_validation import validate_network
from scgraph.pipelines import world_main_roads_and_marnet

print("scgraph imported successfully.")

scgraph imported successfully.


## 1. Zero-RAM File Inspection (`GeoGraph.get_scgraph_info`)

The binary `.scgraph` format allows reading the file header, node/arc counts, and attribution in fractions of a millisecond without loading the graph arrays into memory.

In [2]:
cache_dir = os.path.expanduser("~/.cache/scgraph")
scgraph_file = os.path.join(cache_dir, "world_highways_and_marnet.scgraph")

info = GeoGraph.get_scgraph_info(scgraph_file)
print(f"File: {os.path.basename(scgraph_file)}")
print(f"Format Version: v{info.get('format_version')}")
print(f"Total Nodes: {info['nodes']:,}")
print(f"Total Arcs: {info['arcs']:,}")
print(f"Coordinate Dtype: {info['coordinate_dtype']} (1e-7 deg)")
print(f"Modes: {[m['name'] for m in info.get('modes', [])]}")

File: world_highways_and_marnet.scgraph
Format Version: v1
Total Nodes: 572,009
Total Arcs: 1,689,541
Coordinate Dtype: int32 (1e-7 deg)
Modes: [None]


## 2. Stage 4 Acceptance Certification

We load the global graph into memory (using the accelerated C++ backend) and run the declarative acceptance validation cases defined in `scgraph.pipelines.world_main_roads_and_marnet.VALIDATION_CASES`.

In [3]:
# Load the global intermodal graph
print("Loading global intermodal network...")
world_net = GeoGraph.load_from_scgraph(scgraph_file)
print(f"Network loaded: {len(world_net.nodes):,} nodes ready for queries.\n")

# Define comprehensive global acceptance cases
acceptance_cases = [
    {
        "id": "paris_berlin_highway",
        "origin": (48.8686, 2.3314),
        "destination": (52.5238, 13.3996),
        "expected_km": 1027.7,
        "tolerance": 0.05,
        "comment": "European highway corridor Paris-Berlin",
        "color": "#2ca02c",
    },
    {
        "id": "chicago_houston_highway",
        "origin": (41.8319, -87.7520),
        "destination": (29.8219, -95.3419),
        "expected_km": 1652.3,
        "tolerance": 0.05,
        "comment": "North American Interstate corridor",
        "color": "#9467bd",
    },
    {
        "id": "tokyo_osaka_expressway",
        "origin": (35.6895, 139.6917),
        "destination": (34.6937, 135.5023),
        "expected_km": 460.4,
        "tolerance": 0.05,
        "comment": "Japanese expressway corridor Tomei/Meishin",
        "color": "#1f77b4",
    },
    {
        "id": "rotterdam_nyc_intermodal",
        "origin": (51.95, 4.14),
        "destination": (40.71, -74.01),
        "expected_km": 6150.9,
        "tolerance": 0.05,
        "comment": "Transatlantic intermodal corridor (road + ports + sea)",
        "color": "#ff7f0e",
    },
    {
        "id": "shanghai_la_intermodal",
        "origin": (31.23, 121.47),
        "destination": (34.05, -118.25),
        "expected_km": 10655.9,
        "tolerance": 0.05,
        "comment": "Transpacific intermodal corridor (road + ports + sea)",
        "color": "#d62728",
    },
]

# Execute validation engine
val_result = validate_network(world_net, acceptance_cases, silent=False)
print(f"{val_result.summary()}\n")
assert val_result.is_success, "Acceptance test validation failed!"

print("Detailed Certified Corridor Metrics:")
for detail in val_result.details:
    status = "PASS" if detail["is_success"] else "FAIL"
    print(f"[{status}] {detail['id']:<26}: Measured {detail['measured_km']:>8.1f} km vs Expected {detail['expected_km']:>8.1f} km (origin snap: {detail['snap_origin_km']:.2f} km)")

Loading global intermodal network...
Network loaded: 572,009 nodes ready for queries.

PASSED: 5/5 checks OK (failed: 0)

Detailed Certified Corridor Metrics:
[PASS] paris_berlin_highway      : Measured   1027.7 km vs Expected   1027.7 km (origin snap: 0.47 km)
[PASS] chicago_houston_highway   : Measured   1652.3 km vs Expected   1652.3 km (origin snap: 1.52 km)
[PASS] tokyo_osaka_expressway    : Measured    460.4 km vs Expected    460.4 km (origin snap: 0.61 km)
[PASS] rotterdam_nyc_intermodal  : Measured   6150.9 km vs Expected   6150.9 km (origin snap: 1.60 km)
[PASS] shanghai_la_intermodal    : Measured  10655.9 km vs Expected  10655.9 km (origin snap: 0.11 km)


## 3. Multimodal vs. Monomodal Disconnection Analysis

A pure road network (`world_highways`) cannot connect continental landmasses separated by oceans.
In contrast, `world_highways_and_marnet` bridges continents by routing trucks to commercial seaports and onto global shipping lanes.

In [4]:
road_only_file = os.path.join(cache_dir, "world_highways.scgraph")
road_net = GeoGraph.load_from_scgraph(road_only_file)

p_rdm = {"latitude": 51.95, "longitude": 4.14}  # Port of Rotterdam
p_nyc = {"latitude": 40.71, "longitude": -74.01} # Port of New York

# Test on intermodal network
res_multi = world_net.get_shortest_path(p_rdm, p_nyc, silent=True)
print(f"Intermodal connection (Rotterdam -> NYC): {res_multi['length']:,.1f} km ({len(res_multi['coordinate_path'])} waypoints)")

# Test on road-only network
try:
    road_net.get_shortest_path(p_rdm, p_nyc, silent=True)
    print("Road-only: connected unexpectedly!")
except Exception as err:
    print(f"Road-only network correctly raises {type(err).__name__} (disconnected across Atlantic Ocean).")

Intermodal connection (Rotterdam -> NYC): 6,150.9 km (87 waypoints)
Road-only network correctly raises RuntimeError (disconnected across Atlantic Ocean).


## 4. Interactive Global Cartography (Folium)

We render all certified acceptance corridors on an interactive world map.
- **Green**: Paris ➔ Berlin (European Road Corridor)
- **Purple**: Chicago ➔ Houston (North American Highway)
- **Blue**: Tokyo ➔ Osaka (Asian Expressway)
- **Orange**: Rotterdam ➔ New York (Transatlantic Intermodal)
- **Red**: Shanghai ➔ Los Angeles (Transpacific Intermodal)

In [5]:
# Create global Folium map centered on prime meridian
world_map = folium.Map(location=[30, 0], zoom_start=2, tiles="OpenStreetMap")

# Compute routes and plot
for case in acceptance_cases:
    o, d = case["origin"], case["destination"]
    route = world_net.get_shortest_path(
        {"latitude": o[0], "longitude": o[1]},
        {"latitude": d[0], "longitude": d[1]},
        silent=True,
    )
    length_km = route["length"]
    
    # Draw path
    is_intermodal = "intermodal" in case["id"]
    folium.PolyLine(
        route["coordinate_path"],
        color=case["color"],
        weight=4 if not is_intermodal else 3,
        opacity=0.85,
        dash_array=None if not is_intermodal else "5, 10",
        tooltip=f"{case['id']}: {length_km:,.1f} km",
        popup=f"<b>{case['id']}</b><br>Length: {length_km:,.1f} km<br>{case['comment']}",
    ).add_to(world_map)
    
    # Add start / end markers
    folium.CircleMarker(
        location=o,
        radius=5,
        color=case["color"],
        fill=True,
        fill_color=case["color"],
        fill_opacity=0.9,
        tooltip=f"Origin: {case['id']}",
    ).add_to(world_map)
    folium.CircleMarker(
        location=d,
        radius=5,
        color=case["color"],
        fill=True,
        fill_color=case["color"],
        fill_opacity=0.9,
        tooltip=f"Destination: {case['id']}",
    ).add_to(world_map)

# Display the map
world_map